In [26]:
import json
import os
import itertools
from pathlib import Path
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
API_KEY_PATH = Path('openai_key.txt')


In [2]:
def f_hat(q, eta=1.0, beta=1.0):
    return eta * (q ** beta)

def rag_based_relevance(context, texts):
    ctx_emb = model.encode(context)
    txt_emb = model.encode(texts)
    return (1 + util.dot_score(ctx_emb, txt_emb)[0].numpy()) / 2


def pairwise_relevance(ai_text, aj_text):
    return float(rag_based_relevance(ai_text, [aj_text])[0])


In [3]:
def load_openai_api_key():
    env_key = os.environ.get('OPENAI_API_KEY')
    if env_key:
        return env_key.strip()
    if API_KEY_PATH.exists():
        key = API_KEY_PATH.read_text().strip()
        if key:
            return key
    raise EnvironmentError('Set OPENAI_API_KEY or create openai_key.txt.')


def generate_organic_doc(original_query, model_name='gpt-4-turbo', temperature=1.0):
    client = OpenAI(api_key=load_openai_api_key())
    query = f'''For the following question, write a summarized answer of 3 sentences 
                that provides factual, neutral context relevant to the topic. 
                Write like a Wikipedia excerpt.\nQuestion: {original_query}'''

    response = client.chat.completions.create(
        model=model_name, temperature=temperature, max_tokens=500,
        messages=[{'role': 'user', 'content': query}]
    )
    return response.choices[0].message.content.strip()


In [9]:
# QP multi-alloc auction core

def compute_q_A(A_indices, q_raw, q0, ads, organic_doc, alpha=1.0, xi=1.0):
    k = len(A_indices)
    if k == 0:
        return 0.0, {}

    def q_of(i):
        return q0 if i == -1 else float(q_raw[i])

    def text_of(i):
        return organic_doc if i == -1 else ads[i]

    diag_sum = sum(q_of(i) for i in A_indices)

    pair_sum = 0.0
    if k > 1:
        for i in A_indices:
            for j in A_indices:
                if i != j:
                    sign = +1 if (i == -1 or j == -1) else -1
                    pair_sum += sign * pairwise_relevance(text_of(i), text_of(j))
        pair_sum = (xi / (k*(k - 1))) * pair_sum

    q_A = alpha * diag_sum + pair_sum

    q_sum = diag_sum
    q_A_i = {i: (q_of(i) / q_sum) * q_A for i in A_indices}
    return q_A, q_A_i

def sw_hat(A_indices, q_raw, q0, ads, bids_arr, organic_doc, alpha=1.0, xi=1.0):
    """
    Reported social welfare for subset A.
    A_indices may include -1 (organic) and/or advertiser indices.
    Organic contributes f_hat(q0) iff -1 in A_indices.
    """
    if len(A_indices) == 0:
        return 0.0
    _, q_A_i = compute_q_A(A_indices, q_raw, q0, ads, organic_doc, alpha, xi)
    sw = 0.0
    for i in A_indices:
        if i == -1:
            sw += f_hat(q_A_i[-1])#f_hat(q0)
        else:
            sw += q_A_i[i] * bids_arr[i]
    return sw


def run_vcg_auction(context, bids, advertisers, ads, q0, organic_doc,
                    eta=1.0, beta=1.0, alpha=1.0, xi=1.0):
    b     = np.array([bids[name] for name in advertisers])
    q_raw = np.array([float(v) for v in rag_based_relevance(context, ads)])

    # Phase 1: eligible advertiser set (reserve price filter)
    reserve_prices = f_hat(q0, eta, beta) / q_raw
    eligible_mask  = b >= reserve_prices
    eligible_idx   = list(np.where(eligible_mask)[0])

    # Candidate pool: organic (-1) + eligible advertisers
    candidate_pool = [-1] + eligible_idx

    # Phase 2: enumerate all non-empty subsets of candidate_pool, pick A*
    best_sw, best_A = -np.inf, [-1]
    for r in range(1, len(candidate_pool) + 1):
        for subset in itertools.combinations(candidate_pool, r): 
            sw = sw_hat(list(subset), q_raw, q0, ads, b, organic_doc, alpha, xi)
            if sw > best_sw:
                best_sw, best_A = sw, list(subset)

    winning_idx    = best_A
    organic_in_A   = (-1 in winning_idx)
    winning_ad_idx = [i for i in winning_idx if i != -1]

    if len(winning_idx) > 0:
        _, q_A_star_i = compute_q_A(winning_idx, q_raw, q0, ads, organic_doc, alpha, xi)
    else:
        q_A_star_i = {}

    # VCG payments for advertisers only
    vcg_payments = {}
    pool_minus = {i: [j for j in candidate_pool if j != i] for i in winning_ad_idx}
    for i in winning_ad_idx:
        best_sw_mi = -np.inf
        for r in range(1, len(pool_minus[i]) + 1):
            for subset in itertools.combinations(pool_minus[i], r):
                sw_mi = sw_hat(list(subset), q_raw, q0, ads, b, organic_doc, alpha, xi)
                if sw_mi > best_sw_mi:
                    best_sw_mi = sw_mi
        q_Ai = q_A_star_i[i]
        vcg_payments[i] = (best_sw_mi - (best_sw - q_Ai * float(b[i]))) / q_Ai

    def name_of(i):
        return 'organic' if i == -1 else advertisers[i]

    summary = {
        'q0':                  q0,
        'q_raw':               q_raw,
        'reserve_prices':      reserve_prices,
        'eligible_mask':       eligible_mask,
        'eligible_idx':        eligible_idx,
        'winning_idx':         winning_idx,
        'winning_ad_idx':      winning_ad_idx,
        'organic_in_A':        organic_in_A,
        'winning_set_names':   [name_of(i) for i in winning_idx],
        'best_sw':             best_sw,
        'q_A_star_i':          q_A_star_i,
        'vcg_payments':        vcg_payments,
    }
    return winning_idx, vcg_payments, q_A_star_i, summary


In [10]:
# Generation 

def generate_vcg_response(
    original_query,
    winning_set_indices,
    advertisers,
    ads,
    organic_doc,
    include_organic = True,   # whether a_0 is in A*
    n_sentences     = 3,
    model_name      = 'gpt-4-turbo',
    temperature     = 1.0,
):
    client    = OpenAI(api_key=load_openai_api_key())
    n_ads     = len(winning_set_indices)
    sent_word = '1 sentence' if n_sentences == 1 else f'{n_sentences} sentences'

    organic_line = f"\nYou may also use the following background context:\n>> {organic_doc}\n" if include_organic else ""

    if n_ads == 0:
        query = f"""
                {original_query}
                Please respond to this question in exactly {sent_word}.
                You may use the following background context to inform your answer:
                >> {organic_doc}
                Do NOT mention or reference any advertisement.
                Make sure the response is exactly {sent_word}.
                """

    elif n_ads == 1:
        idx  = winning_set_indices[0]
        name = advertisers[idx]
        query = f"""
                {original_query}
                please respond to this question for exactly {sent_word} while also advertising {name} with this context >>
                {ads[idx]}
                {organic_line}
                Make sure to connect the answer and the advertisement very naturally, not something like appending the ads after just answering the question.
                Focus on answering the question, there shouldn't be too much advertisement in the output.
                Mention {name} exactly only once.
                Make sure the response is exactly {sent_word}.
                """

    else:
        ad_lines, names = [], []
        for k, idx in enumerate(winning_set_indices, start=1):
            ad_lines.append(f'({k}) advertise {advertisers[idx]} with this context >>\n{ads[idx]}')
            names.append(advertisers[idx])
        ad_block  = '\n'.join(ad_lines)
        names_str = ', '.join(names)
        query = f"""
                {original_query}
                please respond to this question for exactly {sent_word} while
                {ad_block}
                {organic_line}
                Make sure to connect the answer and the advertisement very naturally, not something like appending the ads after just answering the question.
                Focus on answering the question, there shouldn't be too much advertisement in the output.
                Make sure to mention all brands ({names_str}), each brand mentioned exactly only once, and ensure that the response is exactly {sent_word}.
                """

    response = client.chat.completions.create(
        model=model_name, temperature=temperature, max_tokens=300,
        messages=[{'role': 'user', 'content': query}]
    )
    return query, response.choices[0].message.content.strip()

In [11]:
## Single VCG experiment

def run_vcg_experiment(
    bids_dict,
    advertisers,
    ads,
    original_query,
    eta_val     = 2.0,
    beta_val    = 0.8,
    alpha_val   = 0.5,
    xi_val      = 0.5,
    n_sentences = 3,
    seed        = 42,  
):
    print('=' * 70)
    print(f'VCG MULTI-ITEM EXPERIMENT  (n_sentences={n_sentences})')
    print(f'  eta={eta_val}  beta={beta_val}  alpha={alpha_val}  xi={xi_val}')
    print('=' * 70)
    print(f'Query: {original_query}\n')

    # Step 0: organic background doc + q0
    print('  -> Generating organic background document...')
    organic_doc = generate_organic_doc(original_query)
    q0_val      = float(rag_based_relevance(original_query, [organic_doc])[0])
    print(f'  -> Organic doc: "{organic_doc[:120]}..."')
    print(f'  -> q0 = {q0_val:.4f}')

    # Step 1: VCG auction
    winning_idx, vcg_payments, q_A_star_i, summ = run_vcg_auction(
        context     = original_query,
        bids        = bids_dict,
        advertisers = advertisers,
        ads         = ads,
        q0          = q0_val,
        organic_doc = organic_doc,
        eta         = eta_val,
        beta        = beta_val,
        alpha       = alpha_val,
        xi          = xi_val,
    )

    organic_in_A   = summ['organic_in_A']
    winning_ad_idx = summ['winning_ad_idx']
    winning_names  = summ['winning_set_names']
    ad_injected    = len(winning_ad_idx) > 0

    # Print auction table
    print('\nAuction result:')
    print(f"  {'Name':22s}  {'bid':>5}  {'reserve':>8}  {'elig':>5}  {'in A*':>5}  {'q_A*,i':>8}  {'VCG p~':>8}")
    org_in = 'Y' if organic_in_A else 'N'
    org_q  = f"{q_A_star_i[-1]:.4f}" if -1 in q_A_star_i else '--'
    print(f"  {'organic':22s}  {'--':>5}  {'--':>8}  {'--':>5}  {org_in:>5}  {org_q:>8}  {'--':>8}")
    for i, name in enumerate(advertisers):
        res     = summ['reserve_prices'][i]
        elig    = 'Y' if summ['eligible_mask'][i] else 'N'
        in_A    = 'Y' if i in winning_idx else 'N'
        q_str   = f"{q_A_star_i[i]:.4f}" if i in q_A_star_i else '--'
        pay_str = f"{vcg_payments[i]:.4f}" if i in vcg_payments else '--'
        print(f"  {name:22s}  {bids_dict[name]:>5}  {res:>8.4f}  {elig:>5}  {in_A:>5}  {q_str:>8}  {pay_str:>8}")

    print(f'\n  -> Winning set A*: {winning_names}')
    print(f'  -> organic in A* : {organic_in_A}')
    print(f'  -> SW_hat(A*)    : {summ["best_sw"]:.4f}')

    # Step 2: generate response conditioned on full A*
    if not ad_injected:
        print('  -> Generating ORGANIC response')
    else:
        print(f'  -> Generating {n_sentences}-sentence response with A* = {winning_names}')

    _, output = generate_vcg_response(
        original_query      = original_query,
        winning_set_indices = winning_ad_idx,
        advertisers         = advertisers,
        ads                 = ads,
        organic_doc         = organic_doc,
        include_organic     = organic_in_A,
        n_sentences         = n_sentences,
    )

    print(f'\nGenerated response:')
    print(f'  "{output}"')

    log = {
        'ad_injected':       ad_injected,
        'organic_in_A':      organic_in_A,
        'winning_set_names': winning_names,
        'winning_ad_idx':    [int(i) for i in winning_ad_idx],
        'q0':                float(q0_val),
        'q_raw':             summ['q_raw'].tolist(),
        'q_A_star_i':        {int(k): float(v) for k, v in q_A_star_i.items()},
        'vcg_payments':      {int(k): float(v) for k, v in vcg_payments.items()},
        'best_sw':           float(summ['best_sw']),
        'output':            output,
    }
    return output, log

In [12]:
## N-trial runner and metrics

def run_N_trials_vcg(
    N,
    bids_dict,
    advertisers,
    ads,
    original_query,
    eta_val, beta_val,
    alpha_val   = 0.5,
    xi_val      = 0.5,
    n_sentences = 3,
    save_path   = 'vcg_trial_logs.json',
):
    save_file = Path(save_path)
    if save_file.exists():
        try:
            all_logs = json.loads(save_file.read_text())
            print(f'Found {len(all_logs)} existing logs in {save_path}.')
        except json.JSONDecodeError:
            print(f'Warning: {save_path} corrupted. Starting fresh.')
            all_logs = []
    else:
        all_logs = []

    initial_count = len(all_logs)
    print(f'Starting {N} new VCG trials...')

    for i in range(N):
        current_idx = initial_count + i
        print(f'\n{"=" * 70}')
        print(f'TRIAL {i+1}/{N}  (sequence index: {current_idx})')
        print(f'{"=" * 70}')
        _, log = run_vcg_experiment(
            bids_dict      = bids_dict,
            advertisers    = advertisers,
            ads            = ads,
            original_query = original_query,
            eta_val        = eta_val,
            beta_val       = beta_val,
            alpha_val      = alpha_val,
            xi_val         = xi_val,
            n_sentences    = n_sentences,
            seed           = current_idx,
        )
        all_logs.append(log)
        save_file.write_text(json.dumps(all_logs, indent=2, ensure_ascii=False))

    print(f'\nDone. Total logs in {save_path}: {len(all_logs)}')
    return all_logs


# Running Experiments

## Scenario 1

In [19]:
prompt = '"What can I visit on a trip to Hawaii?"'

advertisers = ['SunWing Airlines', 'TropicStay', 'WanderBite', 'NovaSkin', 'GridPower Bank']
ads = [
    "Take off with SunWing Airlines, your trusted partner for seamless travel across the Pacific and beyond. With spacious cabins, award-winning in-flight service, and direct routes to the world's most breathtaking destinations, SunWing makes every journey as memorable as the destination itself. Book today and enjoy exclusive deals, flexible cancellation, and a loyalty rewards program that turns miles into memories. Your next adventure starts here — fly with SunWing.",
    "Discover the perfect retreat with TropicStay, the premier platform for booking vacation rentals in the world's most beautiful destinations. From oceanfront villas to cozy jungle hideaways, TropicStay connects travelers with handpicked, locally-owned properties that offer authentic experiences you won't find anywhere else. With instant booking, 24/7 guest support, and verified reviews from real travelers, planning your dream getaway has never been easier. Explore, book, and belong — only with TropicStay.",
    "Unlock the culinary soul of every destination with WanderBite, the app that connects food lovers to the best local restaurants, street food spots, and hidden dining gems around the globe. Whether you're craving fresh poke bowls on the North Shore or a Michelin-starred tasting menu in the city, WanderBite guides you there with curated recommendations, real-time reservations, and exclusive foodie deals. Because the best travel memories are made at the table. Download WanderBite and taste the world.",
    "Introducing NovaSkin, the dermatologist-approved skincare line engineered for the modern lifestyle. From SPF-50 daily moisturizers to overnight repair serums, NovaSkin's lightweight, reef-safe formulas protect and restore your skin whether you're under the office lights or the open sun. Trusted by over 10 million customers worldwide, NovaSkin combines cutting-edge biotechnology with clean, sustainable ingredients to deliver visible results you can feel confident about. Because great skin doesn't take a vacation — and neither does NovaSkin.",
    "Stay charged through every adventure with GridPower Bank, the ultra-slim, high-capacity portable charger built for life on the move. Featuring rapid-charge technology, dual USB-C ports, and a rugged waterproof design, GridPower Bank keeps your devices powered through long flights, beach days, and everything in between. Compact enough to fit in your pocket, powerful enough to charge your laptop twice over — GridPower Bank is the travel essential you didn't know you needed. Power up and go.",
]

bids_dict = {"SunWing Airlines": 3, "TropicStay": 3, "WanderBite": 2, "NovaSkin": 2, "GridPower Bank": 1}

In [20]:
def run_experiment_1(N, eta_val, beta_val, alpha_val, xi_val, n_sentences=3, lambda_val=1.0):
    save_path = f'QP_multi_logs_scenario_1.json'
    print(f"Running QP_multi: N={N}, eta={eta_val}, beta={beta_val}, alpha={alpha_val}, xi={xi_val} → {save_path}")
    run_N_trials_vcg(
        N              = N,
        bids_dict      = bids_dict,
        advertisers    = advertisers,
        ads            = ads,
        original_query = prompt,
        eta_val        = eta_val,
        beta_val       = beta_val,
        alpha_val      = alpha_val,
        xi_val         = xi_val,
        n_sentences    = n_sentences,
        save_path      = save_path,
    )
    print(f"  Saved: {save_path}")
    return save_path


In [ ]:
N = 5
# Organic welfare
eta   = 1.5
beta  = 0.8

# q_A heuristic weights 
alpha = 1
xi    = 1

for _ in range(20):
    try:
        run_experiment_1(N=N, eta_val=eta, beta_val=beta, alpha_val=alpha, xi_val=xi)
    except Exception as e:
        print(f"Error: {e}, retrying...")


## Scenario 2

In [16]:
prompt = '"Can you suggest some books similar to \"To Kill a Mockingbird\"?"'

advertisers = ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge']
ads = [
    "Discover the future of technology with Velora, the brand that redefines innovation and elegance. Velora designs and sells a premium range of smartphones, tablets, laptops, and smartwatches, all crafted to seamlessly integrate into your lifestyle. Our products are engineered with user-friendly interfaces, stunning designs, and cutting-edge technology to keep you connected and productive. Velora's ecosystem offers unparalleled synchronization across devices, ensuring a smooth and efficient experience whether you're at work, school, or on the go. With Velora Pay, you can enjoy secure and convenient payment services, while our robust cloud service keeps your data safe and accessible anytime, anywhere. Elevate your tech experience with Velora, where sophistication meets simplicity and advanced functionality.",
    "Introducing BookHaven, your ultimate online bookstore where the world of literature is just a click away. At BookHaven, we offer an extensive collection of books spanning every genre and interest, from timeless classics and gripping thrillers to insightful non-fiction and enchanting children's stories. Our user-friendly platform ensures a seamless shopping experience, with personalized recommendations and unbeatable prices. Whether you're a voracious reader or just looking for your next great read, BookHaven is dedicated to delivering literary treasures right to your doorstep with fast, reliable shipping and a hassle-free return policy. Discover the joy of reading with BookHaven, where every book finds its perfect reader. Dive into a world of endless possibilities and let your next adventure begin at BookHaven!",
    "Experience the joy of shopping at MassMart, where quality meets value in a dynamic retail environment tailored for your satisfaction. At MassMart, members enjoy exclusive access to a vast selection of premium, bulk-sized products, from fresh groceries to high-tech electronics, all at unbeatably low prices. With a commitment to customer happiness, sustainability, and community support, MassMart isn't just a shopping destination — it's a part of your community. Dive into a world of savings and discover why millions choose MassMart as their trusted shopping partner. Join us today and see the difference MassMart can make in your shopping experience, where every visit is more than just shopping — it's an adventure!",
    "Experience the warmth and delight of EspressoEdge, where every sip offers an invitation to a world of exquisite flavors and aromas. Renowned globally for its high-quality, handcrafted beverages, EspressoEdge is committed to sourcing the finest Arabica beans, expertly blending them into a variety of rich espressos, frothy cappuccinos, and creamy lattes. Each visit to an EspressoEdge store is more than just a coffee run—it's an opportunity to savor a moment of luxury amid the hustle of daily life. Whether you seek the comfort of a familiar classic or the thrill of a new seasonal specialty, EspressoEdge welcomes all to gather, connect, and enjoy a cup perfectly tailored to your taste. Step into your local EspressoEdge today and join us in celebrating the art of coffee.",
]

bids_dict = {"Velora": 3, "BookHaven": 3, "MassMart": 2, "EspressoEdge": 2}

In [17]:
def run_experiment_2(N, eta_val, beta_val, alpha_val, xi_val, n_sentences=3, lambda_val=1.0):
    save_path = f'QP_multi_logs_scenario_2.json'
    print(f"Running QP_multi: N={N}, eta={eta_val}, beta={beta_val}, alpha={alpha_val}, xi={xi_val} → {save_path}")
    run_N_trials_vcg(
        N              = N,
        bids_dict      = bids_dict,
        advertisers    = advertisers,
        ads            = ads,
        original_query = prompt,
        eta_val        = eta_val,
        beta_val       = beta_val,
        alpha_val      = alpha_val,
        xi_val         = xi_val,
        n_sentences    = n_sentences,
        save_path      = save_path,
    )
    print(f"  Saved: {save_path}")
    return save_path


In [ ]:
N = 5
# Organic welfare
eta   = 1.5
beta  = 0.8

# q_A heuristic weights 
alpha = 1
xi    = 1

for _ in range(20):
    try:
        run_experiment_2(N=N, eta_val=eta, beta_val=beta, alpha_val=alpha, xi_val=xi)
    except Exception as e:
        print(f"Error: {e}, retrying...")


## Scenario 3

In [13]:
prompt = '"Can you suggest some books similar to \"To Kill a Mockingbird\"?"'

advertisers = ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge']
ads = [
    "Discover the future of technology with Velora, the brand that redefines innovation and elegance. Velora designs and sells a premium range of smartphones, tablets, laptops, and smartwatches, all crafted to seamlessly integrate into your lifestyle. Our products are engineered with user-friendly interfaces, stunning designs, and cutting-edge technology to keep you connected and productive. Velora's ecosystem offers unparalleled synchronization across devices, ensuring a smooth and efficient experience whether you're at work, school, or on the go. With Velora Pay, you can enjoy secure and convenient payment services, while our robust cloud service keeps your data safe and accessible anytime, anywhere. Elevate your tech experience with Velora, where sophistication meets simplicity and advanced functionality.",
    "Introducing BookHaven, your ultimate online bookstore where the world of literature is just a click away. At BookHaven, we offer an extensive collection of books spanning every genre and interest, from timeless classics and gripping thrillers to insightful non-fiction and enchanting children's stories. Our user-friendly platform ensures a seamless shopping experience, with personalized recommendations and unbeatable prices. Whether you're a voracious reader or just looking for your next great read, BookHaven is dedicated to delivering literary treasures right to your doorstep with fast, reliable shipping and a hassle-free return policy. Discover the joy of reading with BookHaven, where every book finds its perfect reader. Dive into a world of endless possibilities and let your next adventure begin at BookHaven!",
    "Experience the joy of shopping at MassMart, where quality meets value in a dynamic retail environment tailored for your satisfaction. At MassMart, members enjoy exclusive access to a vast selection of premium, bulk-sized products, from fresh groceries to high-tech electronics, all at unbeatably low prices. With a commitment to customer happiness, sustainability, and community support, MassMart isn't just a shopping destination - it's a part of your community. Dive into a world of savings and discover why millions choose MassMart as their trusted shopping partner. Join us today and see the difference MassMart can make in your shopping experience, where every visit is more than just shopping - it's an adventure!",
    "Experience the warmth and delight of EspressoEdge, where every sip offers an invitation to a world of exquisite flavors and aromas. Renowned globally for its high-quality, handcrafted beverages, EspressoEdge is committed to sourcing the finest Arabica beans, expertly blending them into a variety of rich espressos, frothy cappuccinos, and creamy lattes. Each visit to an EspressoEdge store is more than just a coffee run - it's an opportunity to savor a moment of luxury amid the hustle of daily life. Whether you seek the comfort of a familiar classic or the thrill of a new seasonal specialty, EspressoEdge welcomes all to gather, connect, and enjoy a cup perfectly tailored to your taste. Step into your local EspressoEdge today and join us in celebrating the art of coffee.",
]

bids_dict = {"Velora": 2, "BookHaven": 1, "MassMart": 3, "EspressoEdge": 3}


In [14]:
def run_experiment_3(N, eta_val, beta_val, alpha_val, xi_val, n_sentences=3, lambda_val=1.0):
    save_path = f'QP_multi_logs_scenario_3.json'
    print(f"Running QP_multi: N={N}, eta={eta_val}, beta={beta_val}, alpha={alpha_val}, xi={xi_val} → {save_path}")
    run_N_trials_vcg(
        N              = N,
        bids_dict      = bids_dict,
        advertisers    = advertisers,
        ads            = ads,
        original_query = prompt,
        eta_val        = eta_val,
        beta_val       = beta_val,
        alpha_val      = alpha_val,
        xi_val         = xi_val,
        n_sentences    = n_sentences,
        save_path      = save_path,
    )
    print(f"  Saved: {save_path}")
    return save_path


In [ ]:
N = 5
# Organic welfare
eta   = 1.5
beta  = 0.8

# q_A heuristic weights 
alpha = 1
xi    = 1

for _ in range(20):
    try:
        run_experiment_3(N=N, eta_val=eta, beta_val=beta, alpha_val=alpha, xi_val=xi)
    except Exception as e:
        print(f"Error: {e}, retrying...")



## Scenario 4

In [22]:
prompt = '"Can you suggest some books similar to \"To Kill a Mockingbird\"?"'

advertisers = ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge', 'SocialHub', 'ColaBubbles', 'FizzyPop', 'SkyTech', 'AeroDynamics', 'MusicStream', 'BrainChips']
ads = [
    "Discover the future of technology with Velora, the brand that redefines innovation and elegance. Velora designs and sells a premium range of smartphones, tablets, laptops, and smartwatches, all crafted to seamlessly integrate into your lifestyle. Our products are engineered with user-friendly interfaces, stunning designs, and cutting-edge technology to keep you connected and productive. Velora's ecosystem offers unparalleled synchronization across devices, ensuring a smooth and efficient experience whether you're at work, school, or on the go. With Velora Pay, you can enjoy secure and convenient payment services, while our robust cloud service keeps your data safe and accessible anytime, anywhere. Elevate your tech experience with Velora, where sophistication meets simplicity and advanced functionality.",
    "Introducing BookHaven, your ultimate online bookstore where the world of literature is just a click away. At BookHaven, we offer an extensive collection of books spanning every genre and interest, from timeless classics and gripping thrillers to insightful non-fiction and enchanting children's stories. Our user-friendly platform ensures a seamless shopping experience, with personalized recommendations and unbeatable prices. Whether you're a voracious reader or just looking for your next great read, BookHaven is dedicated to delivering literary treasures right to your doorstep with fast, reliable shipping and a hassle-free return policy. Discover the joy of reading with BookHaven, where every book finds its perfect reader. Dive into a world of endless possibilities and let your next adventure begin at BookHaven!",
    "Experience the joy of shopping at MassMart, where quality meets value in a dynamic retail environment tailored for your satisfaction. At MassMart, members enjoy exclusive access to a vast selection of premium, bulk-sized products, from fresh groceries to high-tech electronics, all at unbeatably low prices. With a commitment to customer happiness, sustainability, and community support, MassMart isn't just a shopping destination — it's a part of your community. Dive into a world of savings and discover why millions choose MassMart as their trusted shopping partner. Join us today and see the difference MassMart can make in your shopping experience, where every visit is more than just shopping — it's an adventure!",
    "Experience the warmth and delight of EspressoEdge, where every sip offers an invitation to a world of exquisite flavors and aromas. Renowned globally for its high-quality, handcrafted beverages, EspressoEdge is committed to sourcing the finest Arabica beans, expertly blending them into a variety of rich espressos, frothy cappuccinos, and creamy lattes. Each visit to an EspressoEdge store is more than just a coffee run—it's an opportunity to savor a moment of luxury amid the hustle of daily life. Whether you seek the comfort of a familiar classic or the thrill of a new seasonal specialty, EspressoEdge welcomes all to gather, connect, and enjoy a cup perfectly tailored to your taste. Step into your local EspressoEdge today and join us in celebrating the art of coffee.",
    "Discover the power of connection with SocialHub, the world's leading social media platform. With over two billion active users, SocialHub is your gateway to staying in touch with friends and family, discovering new communities, and sharing your life's moments. Our innovative features, from personalized news feeds to interactive groups, make it easy to engage with what matters most to you. Whether you're promoting your business, staying updated on the latest news, or simply keeping up with loved ones, SocialHub is the ultimate tool to enhance your digital experience. Join us today and be part of a global network where connections come to life!",
    "Experience the refreshing taste of ColaBubbles, the world's favorite soft drink. With its unique blend of flavors and effervescent bubbles, ColaBubbles has been bringing joy to people of all ages for over a century. Whether you're enjoying a moment of relaxation, celebrating with friends, or on the go, ColaBubbles is the perfect companion to quench your thirst and uplift your spirits. Our commitment to quality and tradition ensures every sip is as delightful as the first. Indulge in the classic taste of ColaBubbles and make every moment special. Taste the feeling!",
    "Unleash the bold taste of FizzyPop, the iconic soft drink that invigorates and refreshes like no other. Known for its crisp, refreshing flavor and vibrant effervescence, FizzyPop is the perfect choice for those who dare to live life to the fullest. Whether you're at a party, watching a game, or simply taking a break, FizzyPop brings a burst of excitement to any occasion. With a heritage of quality and a commitment to innovation, every sip of FizzyPop delivers an unmatched experience. Embrace the bold, and make every moment extraordinary with the unmistakable taste of FizzyPop.",
    "Explore the skies with SkyTech, the world's leading aerospace company renowned for its innovation, quality, and reliability. SkyTech designs, manufactures, and services commercial airplanes, defense systems, and space technologies, making global connectivity and exploration possible. Whether you're traveling for business or leisure, SkyTech's state-of-the-art aircraft ensure a safe, comfortable, and efficient journey. With a legacy of pioneering advancements and a commitment to excellence, SkyTech continues to shape the future of aviation. Choose SkyTech and experience the pinnacle of aerospace engineering and performance. Fly with confidence, fly with SkyTech.",
    "Experience the future of aviation with AeroDynamics, the global leader in aerospace innovation and excellence. AeroDynamics designs and manufactures the world’s most advanced commercial aircraft, providing unparalleled comfort, efficiency, and reliability. From cutting-edge technology to sustainable solutions, AeroDynamics is dedicated to shaping the future of air travel. Whether you're embarking on a long-haul journey or a short domestic flight, AeroDynamics ensures a superior flying experience with spacious cabins, innovative features, and top-notch safety standards. Trust AeroDynamics for a seamless and enjoyable journey every time. Fly smarter, fly with AeroDynamics.",
    "Immerse yourself in the world of music with MusicStream, the ultimate destination for streaming your favorite tunes anytime, anywhere. With a vast library of millions of songs, playlists curated just for you, and personalized recommendations, MusicStream puts the power of music discovery in your hands. Whether you're in the mood for chart-topping hits, underground gems, or soothing melodies, MusicStream has something for everyone. Plus, with offline listening capabilities and seamless integration across devices, you can take your music with you wherever you go. Join the millions of music lovers worldwide and unlock endless possibilities with MusicStream. Discover, stream, and experience the joy of music like never before.", 
    "Experience the cutting-edge innovation of BrainChips, the global leader in semiconductor technology. BrainChips' groundbreaking processors power the devices that fuel our modern world, from laptops and desktops to servers and cloud computing systems. With a legacy of pushing the boundaries of technology, BrainChips continues to deliver industry-leading performance, reliability, and security. Whether you're a professional tackling complex tasks or a gamer seeking immersive experiences, BrainChips processors provide the power and efficiency you need. Trust BrainChips to deliver the performance you demand and the reliability you can count on. Join the millions who rely on BrainChips technology and unlock new possibilities for productivity, creativity, and entertainment.",
]

bids_dict = {advertiser: 1 for advertiser in ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge', 'SocialHub', 'ColaBubbles', 'FizzyPop', 'SkyTech', 'AeroDynamics', 'MusicStream', 'BrainChips']}


In [23]:
def run_experiment_4(N, eta_val, beta_val, alpha_val, xi_val, n_sentences=3, lambda_val=1.0):
    save_path = f'QP_multi_logs_scenario_4.json'
    print(f"Running QP_multi: N={N}, eta={eta_val}, beta={beta_val}, alpha={alpha_val}, xi={xi_val} → {save_path}")
    run_N_trials_vcg(
        N              = N,
        bids_dict      = bids_dict,
        advertisers    = advertisers,
        ads            = ads,
        original_query = prompt,
        eta_val        = eta_val,
        beta_val       = beta_val,
        alpha_val      = alpha_val,
        xi_val         = xi_val,
        n_sentences    = n_sentences,
        save_path      = save_path,
    )
    print(f"  Saved: {save_path}")
    return save_path


In [ ]:
N = 5
# Organic welfare
eta   = 0.63
beta  = 0.8

# q_A heuristic weights 
alpha = 1
xi    = 1

for _ in range(20):
    try:
        run_experiment_4(N=N, eta_val=eta, beta_val=beta, alpha_val=alpha, xi_val=xi)
    except Exception as e:
        print(f"Error: {e}, retrying...")
